# MARV on Qwen3-0.6B (Colab T4)

Extract a **vindex** from Qwen3-0.6B, browse what it knows, then **edit a feature
constellation on the live model and measure exactly what broke** — the point of MARV.

Also: a real **weight-space diff** of `Qwen3-0.6B-Base` vs `Qwen3-0.6B` (same
architecture, 28 layers) — git-diff for what post-training moved.

Runtime: **GPU (T4)**. `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
!pip install -q 'transformers>=4.51' accelerate safetensors
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load Qwen3-0.6B

Qwen3 is a plain Llama-style gated FFN (`mlp.gate_proj/up_proj/down_proj` + SiLU),
so `marv.detect_adapter` handles it with no special-casing. QK-norm and the
thinking-mode chat template live in attention / templating — MARV never touches
either (we use raw prompts, not the chat template).


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import marv

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

NAME = 'Qwen/Qwen3-0.6B'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

## Extract the vindex

`extract` copies the FFN gate/down + embed/unembed + final-norm into numpy.
`build_down_meta` precomputes every feature's promoted tokens once (`lm_head @ down`)
so `describe_*` is a lookup afterwards — pure weight math, no forward pass.


In [ ]:
vindex = marv.extract(model, model_name=NAME)
marv.build_down_meta(vindex)
print('bands:', vindex.layer_bands)

# save wherever you like (Drive mount, /content, ...) — MARV never hardcodes a path
vindex.save('/content/qwen3-0.6b.vindex.npz')
print('saved', round(__import__('os').path.getsize('/content/qwen3-0.6b.vindex.npz')/1e6,1), 'MB')

## Browse: `describe_entity`

Embed the entity, KNN its gate features across the knowledge band, label each by
the tokens it promotes. No knowledge-graph pipeline — you get `feature -> tokens`,
not `France --capital--> Paris`.


In [ ]:
for entity in ['France', 'Germany', 'Japan', 'Einstein']:
    print(f'\n=== {entity} ===')
    for row in marv.describe_entity(vindex, tok, entity, k_features=4):
        print('  ', row)

## The constellation, ranked by layer

A fact is not one neuron — it's a weighted pattern across several features in the
knowledge band. `constellation` returns them sorted by similarity; slice `[:n]` for
a minimal set. Below, grouped by layer so you see *where* the fact lives.


In [ ]:
con = marv.constellation(vindex, tok, 'France', per_layer=4)
from collections import defaultdict
by_layer = defaultdict(list)
for r in con:
    by_layer[r.layer].append(r)
for layer in sorted(by_layer):
    rows = by_layer[layer]
    print(f'L{layer}: ' + '  '.join(f'f{r.feature}(sim={r.sim:.2f} {r.tokens[0]!r})' for r in rows))

france_feats = [(r.layer, r.feature) for r in con[:6]]
print('\nconstellation to edit:', france_feats)

## Edit + measure — the collateral-damage table

`suppress` zeros those features on the **live model** via forward hooks (reversible).
`study_edit` runs the battery before/after and reports what moved. Tag your probes so
"broke 3 capitals, held 20 controls" is one line.

- **target**  — what you're trying to change
- **neighbour** — related facts that share the constellation (collateral risk)
- **control** — unrelated, should not move


In [ ]:
P = marv.Probe
battery = [
    P('The capital of France is', 'Paris', ('target','capital')),
    P('Paris is the capital of', 'France', ('target',)),

    P('The capital of Italy is', 'Rome', ('neighbour','capital')),
    P('The capital of Spain is', 'Madrid', ('neighbour','capital')),
    P('The capital of Germany is', 'Berlin', ('neighbour','capital')),
    P('The Eiffel Tower is in', 'Paris', ('neighbour','geography')),
    P('The official language of France is', 'French', ('neighbour','france')),

    P('The capital of Japan is', 'Tokyo', ('control','capital')),
    P('Water is made of hydrogen and', 'oxygen', ('control','science')),
    P('The opposite of hot is', 'cold', ('control','lexical')),
    P('2 + 2 =', '4', ('control','math')),
    P('The sky is', 'blue', ('control',)),
]

rep = marv.study_edit(model, tok, marv.suppress(model, france_feats), battery, device=device)
rep.show()

In [ ]:
# the full list, including everything that held
rep.show(full=True)

## Permanent version: `ablate`

`suppress` is a temporary hook. `ablate` zeros `down_proj[:, f]` in the weights —
affects every path (generate, export). `restore` puts it back.


In [ ]:
before = marv.run_battery(model, tok, battery, device=device)
saved = marv.ablate(model, france_feats)
after = marv.run_battery(model, tok, battery, device=device)
marv.diff_battery(before, after).show()
marv.restore(model, saved)   # undo

## Contextual probe

`describe_entity` queries the bare embedding (fast, weak). `describe_prompt` runs the
real prompt through the model and queries the actual hidden state — stronger matches.
`baseline_prompt` subtracts the templated part to cancel the shared 'massive activation'
direction.


In [ ]:
hits = marv.describe_prompt(
    vindex, model, tok,
    prompt='The capital of France is',
    layers=vindex.band('knowledge'),
    baseline_prompt='The capital of',
    k_features=3, k_tokens=4, device=device,
)
for layer, feats in hits.items():
    for fi, sim, tok_ids, _ in feats:
        words = tok.batch_decode([[int(t)] for t in tok_ids])
        print(f'  L{layer} f{fi} sim={sim:.2f} -> {[w.strip() for w in words]}')

## Weight-space diff: Base vs. post-trained

`Qwen3-0.6B-Base` and `Qwen3-0.6B` are the same architecture (28 layers), so
`marv.diff` compares them feature by feature — which neurons post-training moved,
and whether it changed what they *fire on* (`gate_cos`) or what they *promote*
(`down_cos`). This is MARV's git-diff-for-a-finetune.

_(frees the current model first to keep T4 host RAM happy)_


In [ ]:
del model
import gc; gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base', torch_dtype=torch.float16)
vindex_base = marv.extract(base, model_name='Qwen/Qwen3-0.6B-Base')
marv.build_down_meta(vindex_base)
del base; gc.collect()

deltas = marv.diff(vindex_base, vindex)
print('most-moved features (post-training):')
for d in marv.most_changed(deltas, k=12):
    print(f'  L{d.layer} f{d.feature_idx}: gate_cos={d.gate_cos_sim:+.3f} '
          f'down_cos={d.down_cos_sim:+.3f} norm_ratio={d.gate_norm_ratio:.2f}')

In [ ]:
# label the top movers: what did each promote before vs after?
for d in marv.most_changed(deltas, k=6):
    b, _ = marv.describe_feature(vindex_base, d.layer, d.feature_idx, k=4)
    a, _ = marv.describe_feature(vindex,      d.layer, d.feature_idx, k=4)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'L{d.layer} f{d.feature_idx}:  {bw}  ->  {aw}')

In [ ]:
# which layers absorbed the change
scores = marv.per_layer_score(deltas, metric='mean_topk')
for layer in sorted(scores, key=lambda l: -scores[l])[:8]:
    print(f'  L{layer}: {scores[layer]:.4f}')

## Next steps

- Widen the battery: dozens of `control` probes per edit → the efficacy/specificity
  numbers stabilise.
- Sweep constellation size (`con[:2]` ... `con[:12]`) and plot target-prob vs
  neighbour-degradation — the Pareto frontier of the edit.
- `marv.diff(vindex, quantized_vindex)` — which features 4-bit quantisation breaks.
- `marv.extract_streaming(model_dir)` for a checkpoint too big for T4 RAM.
